<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Agentic RAG
</b></font> </br></p>

---


**Beitrag zum Leitprojekt:** Agentic RAG macht das Evidence Tool des PMO-Assistent selbst kontrolliert entscheidungsfähig — Strategiewahl je Frage, Budget für Retrieval-Aufrufe und ein Quellen-Gate; mehrstufige Suche mit Nachfassen ist die Ausbaustufe. Abgrenzung zu Basic RAG: Dort ruft die Chain den Retriever fest verdrahtet einmal auf; hier **plant** der Agent selbst, ob und mit welcher Strategie gesucht wird — begrenzt durch ein Tool-Budget, nicht durch beliebiges Nachbohren. Im PMO-Assistenten dient das insbesondere der quellengebundenen Meeting-Vorbereitung, Aufgabenverfolgung und Statusberichterstattung.

**Praxis-Transfer:** Mara Vogt bekommt einen Agenten, der Retrieval-Schritte begründet, Quellenlücken erkennt und bei fehlender Evidenz nicht frei ergänzt.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "false"
from langsmith import tracing_context
# Tracing bleibt standardmäßig aus; ausgewählte Referenzläufe werden explizit aktiviert.
os.environ["LANGSMITH_PROJECT"] = "m19_agentic_rag"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS


# 1 | Übersicht
---

Hier geht es um Agentic RAG: Der Agent wählt je Frage die Retrieval-Strategie (breit oder spezifisch) und darf dabei nur ein Budget an Retrieval-Aufrufen verbrauchen. Mehrstufige Suche (Multi-Hop) wäre die naheliegende Erweiterung und wird hier nicht umgesetzt. Für Mara ist entscheidend, dass zusätzliche Flexibilität nicht zu unkontrollierten Tool-Schleifen oder unbelegten Antworten führt.


| Pipeline RAG | Agentic RAG mit Kontrolle |
|---|---|
| Feste Schritte: Retrieve → Generate | Agent wählt die Retrieval-Strategie je Frage |
| Kosten und Latenz gut vorhersagbar | Budget begrenzt die Retrieval-Aufrufe |
| Einfach zu testen | Tool-Protokoll und Quellen-Gate nötig |
| Gut bei ähnlichen Fragen | Gut bei Fragen, die unterschiedliche Suchstrategien brauchen |


<p><font color='darkblue' size="4">
ℹ️ <b>Information:</b>
</font></p>

Dieses Notebook nutzt einen kleinen deterministischen PMO-Korpusauszug. Die RAG-Logik bleibt sichtbar, aber der Lauf hängt nicht von PDF-Parsing, Embeddings oder Modellentscheidungen ab.


In [ ]:
#@markdown <p><font size="4" color='green'>Pipeline RAG vs. Agentic RAG</font></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    subgraph PIPE["Pipeline RAG (immer dieselbe Suche)"]
        P1["Frage"] --> P2["Retrieve einmal"] --> P3["Antwort mit Quellen"]
    end

    subgraph AG["Agentic RAG (Strategie je Frage)"]
        A1["Frage"] --> A2{"Planer"}

        A2 -->|breit suchen| A3["retrieve_broad"]
        A2 -->|spezifisch suchen| A4["retrieve_specific"]

        A3 --> A5["Quellen-Gate"]
        A4 --> A5

        A5 --> A6["Antwort oder Nicht im Korpus"]
    end

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;

    %% Zuweisung
    class P1,A1 input;
    class A2,A5 metric;
    class P3,A6 output;
    style PIPE fill:#f8fafc,stroke:#64748b,stroke-width:2px,stroke-dasharray: 5 5
    style AG fill:#f8fafc,stroke:#16a34a,stroke-width:2px
'''

mermaid(diagram, width=900)


# 2 | Kontrollierter PMO-Korpus
---

Die folgenden Dokumentauszüge simulieren den gemeinsamen PMO-Korpus. Jeder Chunk besitzt eine Quelle, einen Typ und Schlüsselbegriffe. Dadurch lassen sich Retrieval-Entscheidungen deterministisch prüfen.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image as IPImage

BRIEFING_CHUNKS = [
    {
        "source": "protokoll_steuerkreis_2026-04-28.pdf",
        "typ": "risiko",
        "text": "Der Status von Risiko R2 (Datenschutzfreigabe) wird von offen auf mitigiert geändert, weil die Freigabe von Dr. Thomas Brandt vorliegt.",
        "terms": {"risiko", "status", "mitigiert", "datenschutzfreigabe", "freigabe", "steuerkreis"},
    },
    {
        "source": "entscheidungsprotokoll_gesamt.pdf",
        "typ": "entscheidung",
        "text": "Entscheidung D2 (14.04.2026): Eine verwaltete Cloud-Vektordatenbank wird für den produktiven Einsatz genutzt; D1 (ChromaDB) galt nur für den Prototyp.",
        "terms": {"entscheidung", "vektordatenbank", "chromadb", "cloud", "produktiv", "prototyp"},
    },
    {
        "source": "follow_up_tracker.pdf",
        "typ": "aufgabe",
        "text": "Die Rechte- und Rollensteuerung ist offen und vertagt; verantwortlich ist Mara Vogt.",
        "terms": {"aufgabe", "offen", "rechte", "rollensteuerung", "vertagt", "tracker"},
    },
]


def tokenize(text: str) -> set[str]:
    cleaned = text.lower().replace("@", " ").replace("-", " ").replace(".", " ").replace(",", " ")
    return {t for t in cleaned.split() if len(t) > 2}


def score_chunk(query: str, chunk: dict) -> int:
    return len(tokenize(query) & chunk["terms"])

print(f"PMO-Korpusauszug bereit: {len(BRIEFING_CHUNKS)} Chunks")


In [ ]:
def retrieve_broad(query: str, k: int = 3) -> list[dict]:
    ranked = sorted(BRIEFING_CHUNKS, key=lambda c: score_chunk(query, c), reverse=True)
    return [c for c in ranked[:k] if score_chunk(query, c) > 0]


def retrieve_specific(query: str, typ: str) -> list[dict]:
    return [c for c in retrieve_broad(query, k=3) if c["typ"] == typ]


def format_sources(chunks: list[dict]) -> str:
    return "; ".join(sorted({c["source"] for c in chunks}))

print("Retrieval-Funktionen bereit: retrieve_broad, retrieve_specific")


# 3 | Agentic RAG mit Budget und Grounding
---

Der kontrollierte Graph führt pro Lauf einen Retrieval-Schritt mit vom Planer gewählter Strategie aus; ein Budget begrenzt die Retrieval-Aufrufe. Jede Antwort muss durch Quellen gestützt sein oder `Nicht im Korpus.` ausgeben.

**Einordnung:** Das Grounding-Gate ist eine Simulation: `answer_node` ergänzt die Quelle selbst, `grounding_node` prüft nur den Text `Quelle:`. Es ändert in diesem Beispiel daher kein Ergebnis, markiert aber die Stelle, an der eine echte Prüfung stehen würde — als Reflection-Prüfung ohne Rücksprung (siehe Multi-Agent Patterns/Supervisor Pattern), mit LLM-Kritiker in Reflection, Parallel, Hierarchy, Abschnitt 2.


In [ ]:
#@markdown <p><font size="4" color='green'>Kontrolliertes Agentic RAG</font></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TD
    START([START]) --> PLAN["Plan: Fragetyp + Strategie"]
    PLAN --> RET["Retrieve (Budget minus 1)"]
    RET --> GRADE["Relevanz: Treffer vorhanden?"]
    GRADE -->|Treffer| GEN["Antwort mit Quelle"]
    GRADE -->|keine Treffer, Budget leer oder Out-of-Corpus| OOC["Nicht im Korpus"]

    GEN --> CHECK["Grounding-Gate: Quelle vorhanden?"]
    CHECK --> END([END])
    OOC --> CHECK

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef stop fill:#fee2e2,stroke:#dc2626,stroke-width:2px,color:#7f1d1d;

    %% Zuweisung
    class START input;
    class GRADE,CHECK metric;
    class END output;
    class OOC stop;
'''

mermaid(diagram, width=820)


In [ ]:
class AgenticRAGState(TypedDict):
    query: str
    strategy: str
    hops: list[dict]
    documents: list[dict]
    answer: str
    grounded: bool
    budget: int
    log: list[str]


def log_plus(state: AgenticRAGState, eintrag: str) -> list[str]:
    return state.get("log", []) + [eintrag]


def plan_node(state: AgenticRAGState) -> dict:
    frage_tokens = tokenize(state["query"])
    out_of_scope = {"stable", "diffusion", "diagnosis", "medical", "aktien", "wetter"}
    if frage_tokens & out_of_scope:
        strategy = "out_of_scope"
    elif "entscheidung" in frage_tokens or "vektordatenbank" in frage_tokens:
        strategy = "entscheidung"
    elif "risiko" in frage_tokens or "datenschutzfreigabe" in frage_tokens:
        strategy = "risiko"
    else:
        strategy = "broad"
    return {"strategy": strategy, "log": log_plus(state, f"Strategie: {strategy}")}


def retrieve_node(state: AgenticRAGState) -> dict:
    if state["strategy"] == "out_of_scope":
        return {
            "documents": [],
            "hops": state.get("hops", []) + [{"tool": "retrieval_stop", "query": state["query"], "treffer": 0}],
            "log": log_plus(state, "Retrieval gestoppt: Out-of-Corpus"),
        }
    if state["budget"] <= 0:
        return {
            "documents": [],
            "hops": state.get("hops", []) + [{"tool": "retrieval_stop", "query": state["query"], "treffer": 0}],
            "log": log_plus(state, "Retrieval übersprungen: Budget leer"),
        }
    if state["strategy"] == "risiko":
        docs = retrieve_specific(state["query"], "risiko") or retrieve_broad(state["query"], 2)
        tool_name = "retrieve_specific(risiko)"
    elif state["strategy"] == "entscheidung":
        docs = retrieve_specific(state["query"], "entscheidung") or retrieve_broad(state["query"], 2)
        tool_name = "retrieve_specific(entscheidung)"
    else:
        docs = retrieve_broad(state["query"], 2)
        tool_name = "retrieve_broad"
    hop = {"tool": tool_name, "query": state["query"], "treffer": len(docs)}
    return {
        "documents": docs,
        "hops": state.get("hops", []) + [hop],
        "budget": state["budget"] - 1,
        "log": log_plus(state, f"{tool_name}: {len(docs)} Treffer"),
    }


def grade_node(state: AgenticRAGState) -> dict:
    ausreichend = len(state.get("documents", [])) > 0
    return {"grounded": ausreichend, "log": log_plus(state, f"Relevanz: {'ausreichend' if ausreichend else 'keine Treffer'}")}


def answer_node(state: AgenticRAGState) -> dict:
    docs = state.get("documents", [])
    if not docs:
        return {"answer": "Nicht im Korpus.", "grounded": False, "log": log_plus(state, "Antwort: Nicht im Korpus")}
    kern = " ".join(d["text"] for d in docs[:2])
    answer = f"Antwort: {kern} Quelle: {format_sources(docs)}."
    return {"answer": answer, "log": log_plus(state, "Antwort mit Quellen erstellt")}


def grounding_node(state: AgenticRAGState) -> dict:
    grounded = bool(state.get("documents")) and "Quelle:" in state.get("answer", "")
    answer = state["answer"] if grounded else "Nicht im Korpus."
    return {"grounded": grounded, "answer": answer, "log": log_plus(state, f"Grounding: {'ok' if grounded else 'gesperrt'}")}

print("Agentic-RAG-Nodes bereit.")


In [ ]:
agentic_builder = StateGraph(AgenticRAGState)
agentic_builder.add_node("plan", plan_node)
agentic_builder.add_node("retrieve", retrieve_node)
agentic_builder.add_node("grade", grade_node)
agentic_builder.add_node("answer", answer_node)
agentic_builder.add_node("grounding", grounding_node)

agentic_builder.add_edge(START, "plan")
agentic_builder.add_edge("plan", "retrieve")
agentic_builder.add_edge("retrieve", "grade")
agentic_builder.add_edge("grade", "answer")
agentic_builder.add_edge("answer", "grounding")
agentic_builder.add_edge("grounding", END)

agentic_rag_agent = agentic_builder.compile()
print("Agentic-RAG-Graph kompiliert.")


In [ ]:
display(IPImage(agentic_rag_agent.get_graph(xray=True).draw_mermaid_png()))


In [ ]:
with tracing_context(enabled=True):
    run_cfg = {"run_name": "m19_kap3_agentic_rag_grundlauf", "tags": ["agentic-rag", "pmo-briefing"]}
    agentic_result = agentic_rag_agent.invoke(
        {
            "query": "Welche Entscheidung zur Vektordatenbank gilt aktuell, und welche Quelle belegt das?",
            "strategy": "",
            "hops": [],
            "documents": [],
            "answer": "",
            "grounded": False,
            "budget": 2,
            "log": [],
        },
        config={**run_cfg, "run_name": "m19_kap3_agentic_result"},
    )

    mprint("**Agentic-RAG-Log:**")
    mprint("\n".join(f"- {eintrag}" for eintrag in agentic_result["log"]))
    print("Hops:", len(agentic_result["hops"]))
    print(agentic_result["answer"])

# 4 | Budget und Out-of-Corpus
---

Das Budget begrenzt die Retrieval-Aufrufe pro Lauf: Ohne Budget (`budget=0`) wird nicht gesucht, und die Antwort lautet `Nicht im Korpus.`. Out-of-Corpus-Fragen müssen trotz Agentic RAG klar gestoppt werden.

Multi-Hop (mehrere Suchschritte nacheinander) wäre die Erweiterung: Sie braucht eine Rückkante von `grade` zu `retrieve` und eine veränderte Strategie je Hop, damit nicht zweimal dasselbe gefunden wird.

In [ ]:
def run_agentic_rag(query: str, budget: int = 2) -> dict:
    run_cfg = {"run_name": "m19_kap4_agentic_rag_test", "tags": ["test"]}
    return agentic_rag_agent.invoke(
        {
            "query": query,
            "strategy": "",
            "hops": [],
            "documents": [],
            "answer": "",
            "grounded": False,
            "budget": budget,
            "log": [],
        },
        config={**run_cfg, "run_name": "m19_kap4_run_agentic_rag"},
    )

fragen = [
    ("Welchen Status hat das Risiko R2 laut Steuerkreis?", 1),
    ("Was sagt der Korpus zu Stable Diffusion 3?", 1),
    ("Welchen Status hat das Risiko R2 laut Steuerkreis?", 0),  # Budget leer: keine Suche
]
for query, budget in fragen:
    res = run_agentic_rag(query, budget=budget)
    print(f"Frage: {query} (Budget {budget})")
    print(res["answer"])
    print("Letzter Schritt:", res["hops"][-1]["tool"], "| Grounded:", res["grounded"])
    print("---")


# 5 | Vergleich: Pipeline vs. Agentic RAG
---

Agentic RAG ist kein Ersatz für jede Pipeline. Es lohnt sich, wenn die Frage mehrstufig, unklar oder strategieabhängig ist. Bei Standardfragen ist eine feste RAG-Pipeline günstiger und leichter zu testen.


In [ ]:
vergleich = [
    {"kriterium": "Kosten", "pipeline": "vorhersagbar", "agentic": "budgetabhängig"},
    {"kriterium": "Flexibilität", "pipeline": "mittel", "agentic": "hoch"},
    {"kriterium": "Debugging", "pipeline": "einfach", "agentic": "Hop-Protokoll nötig"},
    {"kriterium": "Sicherheit", "pipeline": "feste Gates", "agentic": "Budget und Grounding zwingend"},
]
for row in vergleich:
    print(f"{row['kriterium']}: Pipeline={row['pipeline']} | Agentic={row['agentic']}")


In [ ]:
#@markdown <p><font size="4" color='green'>LangSmith Trace-Analyse</font></p>

try:
    show_trace("m19_agentic_rag", limit=3, show_steps=True)
except Exception as exc:
    print("Trace-Anzeige übersprungen:", type(exc).__name__, str(exc)[:180])


# A | Aufgaben
---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='darkblue' size="4">
✏️  <b>Notiz:</b>
</font></p>

Die Aufgabenstellungen prüfen Agentic RAG als kontrolliertes System: Hop-Protokoll, Grounding und Supervisor-Kombination. Jeder Selfcheck steht direkt unter der zugehörigen Umsetzung.

**Grundlagen**

Ein Hop-Protokoll für Agentic RAG erzeugen. Jeder Retrieval-Schritt soll Tool-Name, Query und Trefferzahl enthalten.


In [ ]:
# Grundlagen: Hop-Protokoll für Agentic RAG erzeugen
# Startpunkt: run_agentic_rag() ist oben bereits definiert

# 1. run_agentic_rag(query, budget=2) aufrufen (grundlagen_result)
# 2. Prüfen: jeder Hop enthält tool, query, treffer

**Aufbau**

Ein Grounding-Gate prüfen. Eine Antwort ohne Quellenhinweis soll gesperrt und zu `Nicht im Korpus.` werden.


In [ ]:
# Aufbau: Grounding-Gate prüfen
# Startpunkt: grounding_node() ist oben bereits definiert

# 1. grounding_node(...) mit einer unbelegten Antwort (kein "Quelle:") aufrufen (grounding_test)
# 2. Prüfen: grounded wird False, Antwort wird zu "Nicht im Korpus."

**Vertiefung**

Agentic RAG mit Supervisor-Logik kombinieren: Die Quellensuche liefert Fakten, Schreib-Schritt formuliert nur bei grounded=True. Kein Schritt darf übersprungen werden.


In [ ]:
# Vertiefung: Agentic RAG mit Supervisor-Logik kombinieren
# Startpunkt: run_agentic_rag() aus Grundlagen

# 1. supervisor_rag(query): recherche-Schritt via run_agentic_rag, schreiben-Schritt nur wenn grounded=True
# 2. Bei grounded=False: sofort stoppen, "Nicht im Korpus." zurueckgeben
# 3. supervisor_rag_result mit Quellensuch- und Schreib-Schritt im Log erzeugen

**Praxis-Transfer-Aufgabe: PMO-Assistent**

Out-of-Corpus-Stopp und Retrieval-Budget.

1. Welche reale Arbeitsfrage löst **dieser** Notebook-Baustein im PMO-Assistent?
2. Welche Eingabedaten oder Dokumente braucht er?
3. Welche Ausgabe sollte er liefern?
4. Welche Risiken oder Grenzen bleiben?
5. Wann wäre Human Review nötig?
6. Ist der passende Lösungsweg hier Prompt, strukturierte Ausgabe, RAG, Tool, Agent oder Workflow?
7. Wie würde die Ausgabe als Meeting-Briefing, Aufgabenübersicht oder Statusreport genutzt und wo ist eine menschliche Prüfung erforderlich?

<p><font color='darkblue' size="4">
✨ <b>Empfehlung:</b>
</font></p>

Agentic RAG nur einsetzen, wenn die Frage wirklich adaptive Suche braucht. Für Standardfragen bleibt Pipeline-RAG günstiger. Für Agentic RAG sind Hop-Protokoll, Retrieval-Budget, Quellen-Gate und Out-of-Corpus-Regel Pflicht.


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [RAG-Pipeline](https://ralf-42.github.io/Agenten/visualisierungen/rag_prozess_visualisierung/)
- [LangGraph](https://ralf-42.github.io/Agenten/visualisierungen/langgraph_explainer/)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [RAG-Konzepte](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/rag-konzepte.html)
- [Context Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/context-engineering.html)
- [Evaluation & Observability](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/evaluation-observability.html)
- [Agent Security](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/agent-security.html)
- [LangGraph Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langgraph-best-practices.html)
